In [1]:
import torch
import torch.nn as nn


# fix seed so that random initialization always performs the same 
torch.manual_seed(13)


# create the model N as described in the question
N = nn.Sequential(nn.Linear(10, 10, bias=False),
                  nn.ReLU(),
                  nn.Linear(10, 10, bias=False),
                  nn.ReLU(),
                  nn.Linear(10, 3, bias=False))

# random input
x = torch.rand((1,10)) # the first dimension is the batch size; the following dimensions the actual dimension of the data
x.requires_grad_() # this is required so we can compute the gradient w.r.t x

t = 0 # target class

epsReal = 0.5  #depending on your data this might be large or small
eps = epsReal - 1e-7 # small constant to offset floating-point erros

# The network N classfies x as belonging to class 2
original_class = N(x).argmax(dim=1).item()  # TO LEARN: make sure you understand this expression
print("Original Class: ", original_class)
assert(original_class == 2)

# compute gradient
# note that CrossEntropyLoss() combines the cross-entropy loss and an implicit softmax function
L = nn.CrossEntropyLoss()
loss = L(N(x), torch.tensor([t], dtype=torch.long)) # TO LEARN: make sure you understand this line
loss.backward()

# your code here
# adv_x should be computed from x according to the fgsm-style perturbation such that the new class of xBar is the target class t above
# hint: you can compute the gradient of the loss w.r.t to x as x.grad
gradient = x.grad
eta = eps * torch.sign(gradient)
adv_x = x - eta

new_class = N(adv_x).argmax(dim=1).item()
print("New Class: ", new_class)
assert(new_class == t)
# it is not enough that adv_x is classified as t. We also need to make sure it is 'close' to the original x. 
print(torch.norm((x-adv_x),  p=float('inf')).data)
assert( torch.norm((x-adv_x), p=float('inf')) <= epsReal)

Original Class:  2
New Class:  0
tensor(0.5000)


In [2]:
import torch
import torch.nn as nn


# fix seed so that random initialization always performs the same 
torch.manual_seed(13)


# create the model N as described in the question
N = nn.Sequential(nn.Linear(10, 10, bias=False),
                  nn.ReLU(),
                  nn.Linear(10, 10, bias=False),
                  nn.ReLU(),
                  nn.Linear(10, 3, bias=False))

# random input
x = torch.rand((1,10)) # the first dimension is the batch size; the following dimensions the actual dimension of the data
x.requires_grad_() # this is required so we can compute the gradient w.r.t x

t = 1 # target class

epsReal = 0.5  #depending on your data this might be large or small
eps = epsReal - 1e-7 # small constant to offset floating-point erros

# The network N classfies x as belonging to class 2
original_class = N(x).argmax(dim=1).item()  # TO LEARN: make sure you understand this expression
print("Original Class: ", original_class)
assert(original_class == 2)

# compute gradient
# note that CrossEntropyLoss() combines the cross-entropy loss and an implicit softmax function
L = nn.CrossEntropyLoss()
loss = L(N(x), torch.tensor([t], dtype=torch.long)) # TO LEARN: make sure you understand this line
loss.backward()

# your code here
# adv_x should be computed from x according to the fgsm-style perturbation such that the new class of xBar is the target class t above
# hint: you can compute the gradient of the loss w.r.t to x as x.grad
gradient = x.grad
adv_x = x - eps * torch.sign(gradient)

# eps tryout
eps_values = [-10, -5, -3, -1, -0.1, -0.3, -0.5, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.8, 1, 2, 3, 5, 7, 10]
for epsilon in eps_values: 
    adv_x = x - epsilon * torch.sign(gradient)
    new_class = N(adv_x).argmax(dim=1).item()
    print("For eps =", epsilon, "new_class = ", new_class, "norm = ", torch.norm((x-adv_x),  p=float('inf')).item())

# iterative fgsm
x_start = x.detach()
alfa_values = [0.001, 0.01, 0.05, 0.07, 0.1, 0.125, 0.150, 0.175, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]
for alfa in alfa_values: 
    adv_x = x_start.clone()
    for i in range(1000):
        adv_x.requires_grad_()
        loss = L(N(adv_x), torch.tensor([t], dtype=torch.long))
        loss.backward()
        eta_iterative_fgsm = alfa * torch.sign(adv_x.grad)
        adv_x = adv_x.detach() - eta_iterative_fgsm
        adv_x = torch.clamp(adv_x, x_start - eps, x_start + eps)
        new_class = N(adv_x).argmax(dim=1).item()
        if new_class == t:
            break
    print("For alfa =", alfa, "new_class = ", new_class, "norm = ", torch.norm((x_start-adv_x),  p=float('inf')).item())
    if new_class == t:
        good_adv_x = adv_x
        
new_class = N(good_adv_x).argmax(dim=1).item()      
print("New Class: ", new_class)
assert(new_class == t)

print(torch.norm((x-good_adv_x),  p=float('inf')).data)
assert( torch.norm((x-good_adv_x), p=float('inf')) <= epsReal)
print(torch.norm(x_start - good_adv_x, p=2))

Original Class:  2
For eps = -10 new_class =  2 norm =  10.0
For eps = -5 new_class =  2 norm =  5.0
For eps = -3 new_class =  2 norm =  3.0
For eps = -1 new_class =  2 norm =  1.0
For eps = -0.1 new_class =  2 norm =  0.10000002384185791
For eps = -0.3 new_class =  2 norm =  0.30000001192092896
For eps = -0.5 new_class =  2 norm =  0.5000000596046448
For eps = 0.05 new_class =  2 norm =  0.050000011920928955
For eps = 0.1 new_class =  2 norm =  0.10000002384185791
For eps = 0.2 new_class =  2 norm =  0.20000004768371582
For eps = 0.3 new_class =  2 norm =  0.30000007152557373
For eps = 0.5 new_class =  2 norm =  0.5000000596046448
For eps = 0.7 new_class =  2 norm =  0.7000000476837158
For eps = 0.8 new_class =  2 norm =  0.8000000715255737
For eps = 1 new_class =  2 norm =  1.0
For eps = 2 new_class =  0 norm =  2.0
For eps = 3 new_class =  0 norm =  3.0
For eps = 5 new_class =  0 norm =  5.0
For eps = 7 new_class =  0 norm =  7.0
For eps = 10 new_class =  0 norm =  10.0
For alfa = 0